In [ ]:
import requests
import json
import os
import urllib.parse  # 한글 인코딩용

def fetch_hanssem_items(search_keyword):
    url = "https://gateway.hanssem.com/hanssem/display-service/api/v1/search/goods-search"
    
    # Referer 헤더용 인코딩
    encoded_keyword = urllib.parse.quote(search_keyword)
    
    params = {
        "page": 1,
        "searchKey": search_keyword,
        "searchType": 0,
        "size": 3,
        "sort": "R"  # ❗️ 'P'(인기순) 대신 'R'(최신순) 사용 (데이터가 나옴)
    }
    headers = {
        "User-Agent": (
            "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
            "AppleWebKit/537.36 (KHTML, like Gecko) "
            "Chrome/141.0.0.0 Safari/537.36"
        ),
        "Accept": "application/json, text/plain, */*",
        "Referer": f"https://store.hanssem.com/search/goods?searchKey={encoded_keyword}",
        "Origin": "https://store.hanssem.com",
        "Accept-Language": "ko-KR,ko;q=0.9,en-US;q=0.8,en;q=0.7",
    }

    resp = requests.get(url, headers=headers, params=params, timeout=10)
    resp.raise_for_status()
    data = resp.json()

    # ▼▼▼▼▼ [수정됨] JSON 경로 수정 ▼▼▼▼▼
    # 'data' -> 'goodsList' (X)
    # 'data' -> 'searchGoodsDataList' -> 'content' (O)
    goods_list = data.get("data", {}).get("searchGoodsDataList", {}).get("content", [])
    
    if not goods_list:
        print(f"경고: API에서 '{search_keyword}'에 대한 상품 목록을 가져오지 못했습니다.")
        return []
        
    results = []
    
    # ▼▼▼▼▼ [수정됨] 항목별 키(key) 이름 수정 ▼▼▼▼▼
    for g in goods_list[:3]:
        
        # 브랜드 (중첩 구조라서 안전하게 .get()으로 접근)
        brand_info = g.get("goodsBrandInfoDto", {})
        brand_name = brand_info.get("brandNm", "브랜드 정보 없음") # 'brandName' -> 'brandNm'

        # 리뷰수 (중첩 구조라서 안전하게 .get()으로 접근)
        eval_info = g.get("goodsEvaluationStatInfoDto", {})
        review_count = eval_info.get("totCnt", 0) # 'reviewCount' -> 'totCnt'

        results.append({
            "브랜드": brand_name,
            "상품명": g.get("gdsNm"),       # 'goodsName' -> 'gdsNm'
            "가격": g.get("dcPrc"),          # 'salePrice' -> 'dcPrc' (할인 가격 기준)
            "할인율": g.get("dcRate"),       # 'discountRate' -> 'dcRate'
            "링크": f"https://store.hanssem.com/goods/{g.get('gdsNo')}" # 'goodsId' -> 'gdsNo'
        })
    return results


if __name__ == "__main__":
    keyword = "침대"
    items = fetch_hanssem_items(keyword)

    if items:
        # ✅ UTF-8 BOM으로 안전하게 파일 저장 (어떤 OS에서도 깨지지 않음)
        file_path = os.path.abspath(f"hanssem_{keyword}_top3.txt")
        with open(file_path, "w", encoding="utf-8-sig", errors="ignore") as f:
            for i, item in enumerate(items, 1):
                f.write(f"{i}. {item['브랜드']} | {item['상품명']}\n")
                f.write(f"   가격: {item.get('가격', 0):,}원 ({item.get('할인율', 0)}% 할인)\n")
                f.write(f"   링크: {item['링크']}\n\n")

        # 콘솔에는 경로만 출력 (ASCII만)
        print(f"\nOK. 결과가 저장되었습니다.\n→ {file_path}")
    else:
        print(f"\n'{keyword}' 키워드에 대한 상품을 찾지 못했습니다.")